# 24_bert_pipeline.ipynb

**12주차 · 2교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`12week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 10셀. 위에서부터 순서대로 실행합니다.

## 0-2. 무엇으로 사전학습했나

**셀 1** — MLM 을 직접 돌려 본다 (30초 확인 ★)

In [ ]:
from transformers import pipeline
fill = pipeline("fill-mask", model="klue/bert-base")
for r in fill("이 영화 정말 [MASK] 재미없었다")[:3]:
    print(f"{r['token_str']:10s} {r['score']:.3f}")

> **관찰 포인트**: 이것이 **사전학습에서 실제로 시킨 과제**입니다. 이 능력이 남아 있기 때문에 감성 분류로 **빠르게 전환**될 수 있습니다.

## 2. 실습 3 — `AutoModel` 출력 뜯어보기 ★

**셀 2** — 백본을 불러 출력 shape 을 본다

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL = "klue/bert-base"
tok   = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()

texts = ["이 영화 정말 재미없었다", "연출이 훌륭하고 배우 연기도 좋았다"]
enc = tok(texts, padding=True, truncation=True, max_length=32, return_tensors="pt")
print("input_ids :", enc["input_ids"].shape)              # (2, T)

with torch.no_grad():
    out = model(**enc)

print("\n출력 키 :", out.keys())
print("last_hidden_state :", out.last_hidden_state.shape)  # ★ (B, T, 768)
print("pooler_output     :", out.pooler_output.shape)      # (B, 768)

> **핵심 ★★**: **`(B, T, 768)`** — 11주차 인코더 블록의 출력 `(B, T, C)` 와 **완전히 같은 모양**입니다. C 가 64 대신 768 일 뿐입니다. *"여러분이 짠 함수의 반환값과 같은 shape 입니다."*

**셀 3** — [CLS] 자리 확인

In [ ]:
tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
print("토큰 :", tokens)
print("0번 자리 :", tokens[0], " ← [CLS] ★")

cls_vec = out.last_hidden_state[:, 0, :]      # (B, 768)  ★ 0번 토큰만 뽑는다
print("\n[CLS] 벡터 :", cls_vec.shape)
print("앞 5개 값 :", cls_vec[0, :5].numpy().round(3))

**셀 4** — [CLS] 벡터로 문장 유사도를 재 본다

In [ ]:
import torch.nn.functional as F
sents = ["이 영화 정말 재미없었다",
         "완전 노잼이었어요",              # 위와 비슷한 뜻
         "연출이 훌륭하고 배우 연기도 좋았다"]   # 반대 뜻
enc2 = tok(sents, padding=True, return_tensors="pt")
with torch.no_grad():
    v = model(**enc2).last_hidden_state[:, 0, :]        # (3, 768)

v = F.normalize(v, dim=-1)
print("문장0 ↔ 문장1 (비슷한 뜻) :", round((v[0] @ v[1]).item(), 3))
print("문장0 ↔ 문장2 (반대 뜻)   :", round((v[0] @ v[2]).item(), 3))

> **관찰 포인트**: 차이가 **크지 않을 수 있습니다.** 사전학습만 된 `[CLS]` 는 문장 유사도용으로 최적화된 게 아니기 때문입니다. **파인튜닝을 해야 이 벡터가 감성에 맞게 정렬됩니다** — 3교시의 동기입니다. (문장 유사도 전용 모델은 3학년 「최신인공지능」의 임베딩·RAG 에서 다룹니다.)

**셀 5** — 분류 헤드가 붙은 버전과 비교 ★

In [ ]:
from transformers import AutoModelForSequenceClassification

clf = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
with torch.no_grad():
    logits = clf(**enc).logits
print("logits :", logits.shape)               # ★ (B, 2)
print(logits)
print("\n확률 :", torch.softmax(logits, dim=-1).numpy().round(3))
print("→ 아직 학습을 안 했으므로 의미 없는 값이다 ★")

**셀 6** — 파라미터 수 · 11주차와 비교

In [ ]:
n = sum(p.numel() for p in model.parameters())
print(f"BERT-base 파라미터 : {n:,} 개")
print(f"내 11주차 블록(C=64,H=8,12층) : 약 60만 개")
print(f"\n층 수 : {model.config.num_hidden_layers} | 은닉 : {model.config.hidden_size}"
      f" | 헤드 : {model.config.num_attention_heads}")

## 4. 실습 4 — Pipeline 3종

**셀 7** — ① 감성 분류

In [ ]:
from transformers import pipeline
import torch
DEV = 0 if torch.cuda.is_available() else -1

sa = pipeline("sentiment-analysis",
              model="matthewburke/korean_sentiment", device=DEV)   # ★ 캐시 확인
for t in ["이 영화 정말 재미없었다", "배우 연기가 최고였어요", "그냥 그랬음"]:
    print(f"{t:25s} → {sa(t)[0]}")

> **관찰 포인트**: *"그냥 그랬음"* 같은 **애매한 문장**을 꼭 넣어 보게 하세요. 확신도(`score`)가 낮게 나오는 것을 보면 **모델이 확률을 내놓는다**는 감각이 생깁니다.

**셀 8** — ② 개체명 인식(NER)

In [ ]:
ner = pipeline("ner", model="Leo97/KoELECTRA-small-v3-modu-ner",
               aggregation_strategy="simple", device=DEV)
text = "봉준호 감독의 기생충은 2019년 칸 영화제에서 황금종려상을 받았다"
for e in ner(text):
    print(f"{e['word']:12s} {e['entity_group']:8s} {e['score']:.3f}")

> **관찰 포인트 ★**: NER 은 **토큰마다** 레이블을 답니다 — `AutoModelForTokenClassification` 이고, `[CLS]` 하나가 아니라 **`(B, T, 클래스수)`** 를 출력합니다. **같은 백본, 다른 헤드**입니다.

**셀 9** — ③ 요약

In [ ]:
summ = pipeline("summarization", model="gogamza/kobart-summarization", device=DEV)
doc = ("영화 기생충은 두 가족의 이야기를 통해 계층 문제를 다룬 작품으로, "
       "2019년 칸 영화제 황금종려상과 2020년 아카데미 작품상을 수상했다. "
       "봉준호 감독은 한국 영화 최초로 아카데미 작품상을 받은 감독이 되었다.")
print(summ(doc, max_length=60, min_length=15)[0]["summary_text"])

**셀 10** — 세 작업의 구조 비교 정리

In [ ]:
print("""
  작업        헤드 종류                          출력 shape
  ─────────────────────────────────────────────────────────────
  감성분류    SequenceClassification (CLS 하나)  (B, 2)
  개체명인식  TokenClassification    (토큰마다)  (B, T, 클래스수)
  요약        Seq2SeqLM (인코더+디코더)          생성 토큰열
                                    └─ 백본은 같은 트랜스포머 ★
""")

> **핵심 ★★**: **백본은 같고 헤드가 다릅니다.** 9주차에 *"백본은 얼려 두고 헤드만 간다"* 고 했던 그 그림이 텍스트에서도 **그대로** 성립합니다.